# `TreeMHN` on iscc data — an R notebook

This notebook runs the real **[TreeMHN](https://github.com/cbg-ethz/TreeMHN)** (Luo, Kuipers &
Beerenwinkel 2023) on a cohort simulated by `iscc`, and scores what it recovers against the
dependency network that was *planted* when the cohort was grown.

It is an **R notebook** — kernel `R (iscc-treemhn)` — and it contains no simulation. The cohort was
generated once by `python validation/make_analysis_data.py --only treemhn`:

| file | what it is | who sees it |
|---|---|---|
| `trees.csv` | one mutation tree per patient, in TreeMHN's own `input_tree_df` layout | the tool |
| `X_presence.csv` | the same cohort as a binary event-presence matrix (what MHN consumes) | comparison |
| `truth_network.json` | the planted epistasis network | **scoring only** |

**What is being asked.** iscc grows each patient over a *known* event-by-event dependency network, so
unlike real data there is a right answer. TreeMHN reads mutation-tree topology, which gives it access
to event **order** — the thing a binary presence matrix throws away.

In [1]:
source("r_preamble.R")
suppressMessages({ library(TreeMHN); library(jsonlite) })

data_dir <- analysis_dir("treemhn")
df    <- read.csv(file.path(data_dir, "trees.csv"))
truth <- fromJSON(file.path(data_dir, "truth_network.json"))

n        <- max(df$Mutation_ID)          # number of real events (the root is 0)
patients <- unique(df$Patient_ID)
cat(sprintf("cohort: %d patients, %d events, %d tree rows\n", length(patients), n, nrow(df)))
cat(sprintf("planted interaction strength: %.2f over %d interactions\n",
            truth$interaction_strength, truth$epistasis_params$n_interactions))
head(df, 8)

               Welcome to the TreeMHN package!

This package is developed by the Computational Biology Group
of ETH Zurich and the Swiss Institute of Bioinformatics (SIB).

Please cite the following paper when using this package:
https://www.nature.com/articles/s41467-023-39400-w

For any questions, please contact niko.beerenwinkel@bsse.ethz.ch



cohort: 16 patients, 4 events, 59 tree rows


planted interaction strength: 2.00 over 2 interactions


,Patient_ID,Tree_ID,Node_ID,Mutation_ID,Parent_ID,n_cells
,<int>,<int>,<int>,<int>,<int>,<int>
1,1,1,1,0,1,70
2,1,1,2,4,1,49
3,1,1,3,2,2,17
4,1,1,4,1,1,1
5,2,2,1,0,1,123
6,2,2,2,2,1,6
7,2,2,3,4,1,5
8,2,2,4,3,1,1


## Fit

`input_tree_df` is TreeMHN's own constructor, and `learn_MHN` returns the estimated Θ — a matrix
whose off-diagonal entries are the promoting (positive) or inhibiting (negative) effects of one event
on another's rate.

In [2]:
tree_df <- df[, c("Patient_ID", "Tree_ID", "Node_ID", "Mutation_ID", "Parent_ID")]
tree_obj <- input_tree_df(n = n, tree_df = tree_df,
                          patients = as.character(patients),
                          mutations = paste0("E", seq_len(n) - 1))

Theta <- learn_MHN(tree_obj, gamma = 0.5, verbose = FALSE, return_Theta_only = TRUE)
rownames(Theta) <- colnames(Theta) <- paste0("E", seq_len(n) - 1)
round(Theta, 3)

,E0,E1,E2,E3
E0,-1.687,-1.081,-1.028,-0.450
E1,-1.323,-0.597,-2.109,-0.547
E2,-1.119,-1.928,-0.712,-2.537
E3,-2.038,-2.847,-1.678,0.223


## What the diagonal and off-diagonal mean

The **diagonal** is each event's baseline rate — how often it happens on its own. The
**off-diagonal** `Theta[i, j]` is the effect of event *j* on event *i*'s rate: positive means *j*
promotes *i*, negative means it suppresses it. The planted network lives in the off-diagonal, so that
is what we score.

A caution this benchmark exists to make concrete: iscc's epistasis acts on **fitness** — how large the
clones carrying a combination grow — not on the **rate** at which events arise. TreeMHN estimates
rates. So a recovered off-diagonal near zero is not necessarily a failure of the tool; it can mean the
planted effect never expressed itself as an ordering signal at all.

In [3]:
off <- Theta[row(Theta) != col(Theta)]
cat(sprintf("off-diagonal entries: %d\n", length(off)))
cat(sprintf("  mean %.3f   sd %.3f   max |value| %.3f\n", mean(off), sd(off), max(abs(off))))
# Label by SIGN, not by position: if every off-diagonal is negative there is no promoting pair at
# all, and calling max(off) "strongest promoting" would misdescribe the fit.
cat(sprintf("  largest off-diagonal:  %+.3f  (%s)\n", max(off),
            ifelse(max(off) > 0, "promoting", "still inhibiting — NO promoting pair was found")))
cat(sprintf("  smallest off-diagonal: %+.3f  (strongest inhibition)\n", min(off)))

cat("\nbaseline rates (diagonal):\n")
print(round(diag(Theta), 3))

cat("\nRead this against the caveat above: iscc's planted epistasis acts on FITNESS, not on event\n")
cat("RATE, and TreeMHN estimates rates. Near-zero off-diagonals here are evidence about WHICH\n")
cat("observable carries the signal, not simply about the tool's accuracy.\n")

off-diagonal entries: 12


  mean -1.557   sd 0.758   max |value| 2.847


  largest off-diagonal:  -0.450  (still inhibiting — NO promoting pair was found)


  smallest off-diagonal: -2.847  (strongest inhibition)



baseline rates (diagonal):


    E0     E1     E2     E3 
-1.687 -0.597 -0.712  0.223 



Read this against the caveat above: iscc's planted epistasis acts on FITNESS, not on event


RATE, and TreeMHN estimates rates. Near-zero off-diagonals here are evidence about WHICH


observable carries the signal, not simply about the tool's accuracy.
